# SPEI Fraud Detection - Data Exploration



This notebook explores synthetic SPEI transactions dataset built for a fraud detection project.
SPEI is México's real-time payment system.

The dataset consists of transactions injected with fraud across five different typologies:
- App Fraud
- Account Takeover (ATO)
- Mule-to-mule
- Smurfing
- Business Impersonation

This notebooks goal is to understand the dataset's structure, content and feature distribution before training a classifier ML model.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.features.load_data import load_fct_features, load_fraud_typology

pd.set_option('display.max_columns', 50)

## Load Data

In [ ]:
df = load_fct_features(refresh=True)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
from src.features.load_data import load_dim_accounts
dim = load_dim_accounts(refresh=True)
print(f"Dim shape: {dim.shape}")

In [ ]:
typology_df = load_fraud_typology(refresh=True)
print(f'Shape: {typology_df.shape}')

In [ ]:
df = pd.merge(df, typology_df, on = 'transaction_id', how = 'left')

In [ ]:
df['fraud_typology'].value_counts(dropna=False)

## Class Distribution

In [ ]:
# Overall fraud rate
fraud_rate = df['is_fraud'].mean()*100
print(f'Overall fraud rate: {fraud_rate: .2f}%')
print(f"Fraud count: {df['is_fraud'].sum():,}")
print(f"Legitimate count: {(~df['is_fraud']).sum()}")

#per-typology count
print("Per-typology distribution:")
print(df['fraud_typology'].value_counts(dropna=False))

## Temporal Structure

In [ ]:
#Date range
print(f"Date range: {df['transaction_date'].min()} to {df['transaction_date'].max()}")
print(f"Days span: {(df['transaction_date'].max() - df['transaction_date'].min()).days}")

In [ ]:
# Daily transaction counts, overlaid with fraud rate
df['date'] = pd.to_datetime(df['transaction_date']).dt.date
daily = df.groupby('date').agg(
    total=('transaction_id', 'count'),
    fraud=('is_fraud', 'sum')
).reset_index()
daily['fraud_rate'] = daily['fraud'] / daily['total']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize = (14,6), sharex = True)
ax1.plot(daily['date'], daily['total'], color='steelblue')
ax1.set_ylabel('Total transactions per day')
ax1.set_title('SPEI transaction volume over time')

ax2.plot(daily['date'], daily['fraud_rate'] * 100, color='crimson')
ax2.set_ylabel('Fraud Rate percentage')
ax2.set_xlabel('Date')
ax2.axhline(y=1.47, color='gray', linestyle='--', label='Overall average')
ax2.legend()

plt.tight_layout()
plt.show()


## Temporal Split Preview
Split into 80/10/10 by date and evaluate typology counts in each split

In [ ]:
df_sorted = df.sort_values('transaction_date').reset_index(drop = True)

n = len(df_sorted)
train_end = int(n * .8)
val_end = int(n * .9)

train = df_sorted.iloc[:train_end]
val = df_sorted.iloc[train_end:val_end]
test = df_sorted.iloc[val_end:]

print(f"Train: {len(train):,} rows from {train['transaction_date'].min()} to {train['transaction_date'].max()}")
print(f"Val: {len(val):,} rows from {val['transaction_date'].min()} to {val['transaction_date'].max()}")
print(f"Test: {len(test):,} rows from {test['transaction_date'].min()} to {test['transaction_date'].max()}")

In [ ]:
#Per typology counts across splits
print('Per typology counts per split:')

split_counts = pd.DataFrame({
    'train': train['fraud_typology'].value_counts(dropna=False),
    'val': val['fraud_typology'].value_counts(dropna=False),
    'test': test['fraud_typology'].value_counts(dropna=False),
}).fillna(0).astype(int)

print(split_counts)
print("Train fraud rate:", f"{train['is_fraud'].mean() * 100:.2f}%")
print("Val fraud rate:", f"{val['is_fraud'].mean() *100:.2f}%")
print("Test fraud rate:", f"{test['is_fraud'].mean() *100:.2f}%")

## Feature Distribution

In [ ]:
# NULL rates
null_rates = df.isnull().mean() * 100
print("NULL rates by column:")
print(null_rates.sort_values(ascending=False).round(2))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (14,4))

#Linear scale
ax1.hist(df['amount'], bins=100, range=(0,30000), color='steelblue')
ax1.set_title('Amount distribution (linear)')
ax1.set_xlabel('Amount(MXN)')
ax1.set_ylabel('Count')

#Log scale
log_bins = np.logspace(0, 7, 100)
ax2.hist(df['amount'], bins=log_bins, color='steelblue')
ax2.set_xscale('log')
ax2.set_title('Amount distribution (log)')
ax2.set_xlabel('Amount(MXN)')

plt.tight_layout()
plt.show()

In [ ]:
features_to_compare = [
    'amount',
    'hour',
    'is_off_hours',
    'sender_txn_count_24h',
    'sender_distinct_receivers_as_of',
    'is_new_beneficiary',
    'near_threshold_flag',
    'pair_txn_count',
    'sender_kyc_tier',
]
comparison = df.groupby('is_fraud')[features_to_compare].mean().T
comparison.columns = ['Legitimate', 'Fraud']
comparison['Ratio'] = (comparison['Fraud'] / comparison['Legitimate']).round(2)
print(comparison.round(2))

In [ ]:
# Amount median vs mean
amount_stats = df.groupby('is_fraud')['amount'].agg(['mean', 'median']).round(2)
amount_stats.index = ['Legitimate', 'Fraud']
amount_stats['mean_ratio'] = amount_stats['mean'] / amount_stats.loc['Legitimate', 'mean']
amount_stats['median_ratio'] = amount_stats['median'] / amount_stats.loc['Legitimate', 'median']
print(amount_stats)

Amount alone is not a strong fraud signal. The median amount of fraud (8.4K MXN) is close to the median of legitimate transactions (8.7K MXN). Matching sophisticated attacks designed to blend in with normal transactions.

## Realism Verification

### Transaction amount by sender tier

KYC tier corresponds to how much personal information the account owner provided when creating the account (tier 1 = minimal, tier 4 = full verification).

In [ ]:
legit = df[df['is_fraud'] == False]
amount_by_tier = legit.groupby('sender_kyc_tier')['amount'].agg(['median', 'count']).round(2)
amount_by_tier.columns = ['Median amount (MXN)', 'Transaction count']
print(amount_by_tier)

### Off-hours concentration for account takeover

Account takeover (ATO) fraud is often attempted during off-hours when victims are less likely to notice. The dataset marks transactions from 10 PM to 5 AM as off-hours.

In [ ]:
legit_rate = df[df['is_fraud'] == False]['is_off_hours'].mean() * 100
fraud = df[df['is_fraud'] == True]
off_hours_by_type = (fraud.groupby('fraud_typology')['is_off_hours'].mean() * 100).round(1)

print(f"Legitimate off-hours rate: {legit_rate:.1f}%\n")
print("Off-hours rate by fraud typology:")
print(off_hours_by_type)

### Smurfing structuring pattern

Smurfing breaks large amounts into smaller transactions to stay below reporting thresholds. In México, SPEI transactions over ~13,200 are more likely to be reviewed by the CNBV (Comisión Nacional Bancaria y de Valores), making smurfing transactions pool together under that threshold.

In [ ]:
legit_rate = df[df['is_fraud'] == False]['near_threshold_flag'].mean() * 100
fraud = df[df['is_fraud'] == True]
near_threshold_by_type = (fraud.groupby('fraud_typology')['near_threshold_flag'].mean() * 100).round(1)

print(f"Legitimate near-threshold rate: {legit_rate:.1f}%\n")
print("Near-threshold rate by fraud typology:")
print(near_threshold_by_type)

### Established-pair fraud

Most fraud transactions go to new beneficiaries. The dataset was made to include a small share of fraud accross established pairs so the model doesn´t rely on new-beneficiary status as a single, perfect signal.

In [ ]:
fraud = df[df['is_fraud'] == True]
established_rate = fraud.groupby('fraud_typology').apply(
    lambda g: (g['pair_txn_count'] > 0).mean() * 100
).round(1)

print("Established-pair rate by fraud typology:")
print(established_rate)